<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c40-header.png" alt="Nextia Learning · Experiments and Causal Impact: Did Our Change Work?" width="100%">

# Solution: Difference-in-differences

**[Difference-in-differences](https://learning.nextia-ai.com/courses/experiments/m05/difference-in-differences/)** · Experiments and Causal Impact: Did Our Change Work? · Module 5, lesson 1 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** estimate the effect of free delivery in Quebec, which started without a test, by comparing Quebec's change with Ontario's change, check the parallel-trends assumption, and run the regression with errors clustered by area.

| | |
|---|---|
| **Time** | About 40 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | Regression with dummy variables, from the [mathematics course](https://learning.nextia-ai.com/courses/math/). |
| **Tested** | Python 3.12 with pandas 3.1.1, SciPy 1.15.3, statsmodels 0.15.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/experiments/m05/difference-in-differences/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C40/M05-L01-difference-in-differences/difference-in-differences-solution.ipynb).

## Setup: install the packages

Run the next cell first. It installs the versions that the course uses: pandas 3.1.1, SciPy 1.15.3 and statsmodels 0.15.0, and matplotlib 3.11.2 for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with these versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 scipy==1.15.3 statsmodels==0.15.0 matplotlib==3.11.2
import importlib.metadata as _md
for _name in ["pandas", "scipy", "statsmodels", "matplotlib"]:
    print(_name, _md.version(_name))

## Setup: get the practice data

Run the next cell. It downloads the simulator's data for this lesson (less than 3 MB) into a folder `larkfield-experiments/data`, and checks each file's checksum. It needs an internet connection and no account.

You should see `Ready: 3 files in data/.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-experiments` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield data is synthetic: the course simulator made it. It has no real people. Each folder has `truth.json`: the true effects that the simulator knows. A real test never has this file.

In [ ]:
# Setup: download the course data into larkfield-experiments/data/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

DATA = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C40/data/"
FILES = {
    "regions/areas_weekly.csv": "d5b4a20ff197705975d98a547a0335f9648e2a302f2090c32731e1586153a3ea",
    "regions/provinces_weekly.csv": "2e86c748e3f53a799e864e1949f8541f7feefedf1be0ec26d79751340ba71238",
    "regions/truth.json": "c1e0f29839d7451fb97d553553056a28e78c22aed5cf7e9f927ebbd559aada08"
}
LOCAL = os.environ.get("NX_DATA_LOCAL")  # authors only: a local copy of the labs data
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("larkfield-experiments").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    if LOCAL:
        shutil.copy(Path(LOCAL) / name, path)
    else:
        download(DATA + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-experiments and run again.")
os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in data/.")

## Setup: helpers

The next cell loads pandas as `pd`, NumPy as `np`, SciPy's `stats` and statsmodels' formula interface as `smf`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `truth(name)` reads a scenario's `truth.json`: the simulator's true effects.

Run the cell. It prints nothing.

In [ ]:
# Helpers: checks that do not show the answer, and the simulator's truth.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
pd.set_option("display.precision", 4)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def truth(name):
    """The simulator's true effects for a scenario (a real test never has this)."""
    return json.loads(open(f"data/{name}/truth.json").read())

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Quebec and Ontario, week by week

Free delivery from 50 dollars started in Quebec on Monday 2 June 2025. Ontario kept the old rule. The next cell draws both provinces' weekly sales.

> **Predict.** Will you see a jump in Quebec on 2 June?

In [ ]:
import matplotlib.pyplot as plt

prov = pd.read_csv("data/regions/provinces_weekly.csv", parse_dates=["week"])
wide = prov.pivot(index="week", columns="province", values="revenue")
ax = (wide[["QC", "ON"]] / 1000).plot(figsize=(11, 4))
ax.axvline(pd.Timestamp("2025-06-02"), color="grey", linestyle="--")
ax.set_ylabel("weekly sales (thousand dollars)")
plt.show()

The season moves both lines a lot more than the change does. A before-and-after comparison of Quebec alone would mix the season with the effect.

## 2. The 2 × 2 table

Difference-in-differences: (Quebec after − Quebec before) − (Ontario after − Ontario before). On the log of sales, the result is a relative change. We use 26 weeks on each side.

In [ ]:
areas = pd.read_csv("data/regions/areas_weekly.csv")
areas["post"] = (areas["week"] >= "2025-06-02").astype(int)
win = areas[(areas.week >= "2024-12-02") & (areas.province.isin(["QC", "ON"]))]
table = np.log(win.groupby(["province", "post", "week"]).revenue.sum()).groupby(["province", "post"]).mean().unstack()
print(table.round(4))
did_hand = (table.loc["QC", 1] - table.loc["QC", 0]) - (table.loc["ON", 1] - table.loc["ON", 0])
print(f"difference-in-differences: {np.expm1(did_hand):+.1%}")

> **Check.** You should see `difference-in-differences: +5.7%`.

## 3. The regression, with errors clustered by area

The same idea as a regression on the 24 delivery areas of the two provinces: an area effect, a week effect, and the effect of being in Quebec after the change. The weeks of one area are not independent, so the errors are **clustered by area**.

In [ ]:
d = areas[areas.province.isin(["QC", "ON"])].copy()
d["treated"] = (d.province == "QC").astype(int)
d["log_revenue"] = np.log(d.revenue)
m = smf.ols("log_revenue ~ treated:post + C(area) + C(week)", data=d).fit(
    cov_type="cluster", cov_kwds={"groups": d["area"].astype("category").cat.codes})
b, se = m.params["treated:post"], m.bse["treated:post"]
print(f"effect {np.expm1(b):+.2%}, 95% interval {np.expm1(b - 1.96 * se):+.2%} to {np.expm1(b + 1.96 * se):+.2%}")
print(f"true effect (simulator): {truth('regions')['effect']['relative']:+.1%}")

> **Check.** You should see `effect +5.28%, 95% interval +4.61% to +5.96%` and `true effect (simulator): +6.0%`.

The interval is narrow, and its upper end is just below the truth. Over 50 seeds of this scenario, the course measured that this comparison misses the truth almost every time (it finds about +3% on average). Difference-in-differences assumes that Quebec, without the change, would have moved **like Ontario**. In this simulator, Quebec shares a slow eastern trend with the Atlantic provinces, and Ontario does not.

## 4. Check parallel trends before the change

Before the change, nothing differed between the provinces except the trends. A **placebo test** pretends the change happened 26 weeks earlier, using only the weeks before the real change.

In [ ]:
pre = d[d.week < "2025-06-02"].copy()
pre["post"] = (pre.week >= "2024-12-02").astype(int)
mp = smf.ols("log_revenue ~ treated:post + C(area) + C(week)", data=pre).fit(
    cov_type="cluster", cov_kwds={"groups": pre["area"].astype("category").cat.codes})
print(f"placebo effect {np.expm1(mp.params['treated:post']):+.1%}, p = {mp.pvalues['treated:post']:.3f}")

> **Check.** You should see `placebo effect -0.7%, p = 0.072`.

A placebo effect near zero is a good sign, but not a proof: here it is small, and the real gap is also small. The lesson page shows the trend gap in a picture.

## 5. Your turn: all provinces as the comparison

> **Your turn.** Run the same regression with **all 60 areas** (Quebec against the 9 other provinces). Put the effect, `np.expm1` of the coefficient, rounded to 4 decimals, in `did_all`.

In [ ]:
da = areas.copy()
da["treated"] = (da.province == "QC").astype(int)
da["log_revenue"] = np.log(da.revenue)
ma = smf.ols("log_revenue ~ treated:post + C(area) + C(week)", data=da).fit(
    cov_type="cluster", cov_kwds={"groups": da["area"].astype("category").cat.codes})
did_all = round(float(np.expm1(ma.params["treated:post"])), 4)
print(did_all)

In [ ]:
expect('did_all', did_all, 0.0551)

With all provinces, the comparison group shares more of Quebec's eastern trend, and the estimate moves closer to the truth. Choosing the comparison is the hard part of the method: the next lesson builds the comparison on purpose.

## Next

The [lesson page](https://learning.nextia-ai.com/courses/experiments/m05/difference-in-differences/) has the diagrams and the knowledge checks that count toward your certificate. In the project: `sim run --scenario regions`, then `xp did --placebo` and `xp did --control all`.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Synthetic control](https://learning.nextia-ai.com/courses/experiments/m05/synthetic-control/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/experiments/m05/difference-in-differences/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.